# Import Librares

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import RFE

from sklearn.ensemble import RandomForestClassifier

# Import Dataset using read_csv method

In [ ]:
url ="https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/refs/heads/master/data/Telco-Customer-Churn.csv" 

df = pd.read_csv(url)

In [ ]:
#Set option to display all rows and column

pd.set_option("display.max_rows",None) #load all columns as is
pd.set_option("display.max_columns",None) #load all columns as is

#View the top 5 rows of dataset
df.head()

In [ ]:
#Checking information about data set
df.info()

In [ ]:
df.shape

In [ ]:
df['gender'].value_counts()

In [ ]:
df['gender'] = df['gender'].map({"Female":0, "Male":1})

In [ ]:
df['gender'].value_counts()

In [ ]:
df['Churn'].value_counts()

In [ ]:
df['Churn'] = df['Churn'].map({"No":0, "Yes":1})

In [ ]:
df['Churn'].value_counts()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')

In [ ]:
df.isnull().sum() # this is bad

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.head(2)

In [ ]:
df.columns

In [ ]:
numerical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 
                      'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                      'DeviceProtection','TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 
                      'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

In [ ]:
for col in numerical_features:
    if df[col].dtype == "object":
        df[col] = pd.factorize(df[col])[0]

In [ ]:
numerical_features = [f for f in numerical_features if f in df.columns]
numerical_features

In [ ]:
X = df[numerical_features]

y = df['Churn']

In [ ]:
X.head()

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20, stratify=y,random_state=42)

In [ ]:
rf = RandomForestClassifier(n_estimators=200)

rfe = RFE(estimator=rf, n_features_to_select=5) #6,7,8


In [ ]:
rfe.fit(X_train,y_train) #use only train data

In [ ]:
rfe.support_

In [ ]:
X.columns[rfe.support_]

In [ ]:
selected_features = [f for f,sel in zip(numerical_features,rfe.support_) if sel] 

In [ ]:
selected_features

In [ ]:
X_train_rfe = rfe.transform(X_train)
X_test_rfe = rfe.transform(X_test)

In [ ]:
rf_final = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf_final.fit(X_train_rfe, y_train)

In [ ]:
importances = pd.DataFrame({'feature':selected_features, 'importance':rf_final.feature_importances_})

In [ ]:
importances = importances.sort_values('importance',ascending=False)

importances

In [ ]:
import seaborn as sns

sns.barplot(x='importance',y='feature',data=importances)

In [ ]:
selected_features

In [ ]:
from sklearn.preprocessing import StandardScaler

#model building

X = df[selected_features]

y = df['Churn']

In [ ]:
X_train,X_test, y_train, y_test = train_test_split(X,y, test_size=0.20)

In [ ]:
scaler = StandardScaler()

In [ ]:
import pickle

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler,f)

In [ ]:
import tensorflow as tf

from tensorflow.keras import layers, models

In [ ]:
def create_model():
    model = models.Sequential([
        layers.Dense(64, activation='relu',input_shape = (5,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16,activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy']) 
    return model

In [ ]:
ann_model = create_model()

ann_model.summary()


In [ ]:
history = ann_model.fit(X_train_scaled, y_train,epochs=50,batch_size=32,validation_split=0.20,)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))

plt.plot(history.history['loss'], label='Training Loss')

plt.plot(history.history['val_loss'], label='Validation Loss')

plt.grid(alpha=0.4)

plt.legend()

plt.xlabel("Epoch")

plt.ylabel("Loss")



In [ ]:

plt.figure(figsize=(8,4))

plt.plot(history.history['accuracy'], label='Training accuracy')

plt.plot(history.history['val_accuracy'], label='Validation accuracy')

plt.grid(alpha=0.4)

plt.legend()

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

In [ ]:
!pip install keras-tuner

In [ ]:
import keras_tuner as kt

In [ ]:
def build_model(hp):
    
    model = models.Sequential()
    
    n_layers = hp.Int('n_layers',min_value=1,max_value=4, step=1)
    
    for i in range(n_layers):
        model.add(layers.Dense(units = hp.Int(f"units_{i}", min_value=16, max_value=256, step=16), 
                               activation = hp.Choice(f"activation_{i}", values=['relu','tanh','elu']), 
                               kernel_regularizer = tf.keras.regularizers.l2(hp.Float(f"lr_{i}", min_value=1e-5, 
                                                    max_value=1e-2, sampling='log')))) 
        model.add(layers.Dropout(rate = hp.Float(f"dropout_{i}", min_value=0.0, max_value=0.5, step=0.1)))
        model.add(layers.Dense(1, activation='sigmoid'))
    
    #learning rate and oiptimser
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log') 
    optimizer = hp.Choice('optimizer',values=['adam','rmsprop','sgd'])
    
    if optimizer == 'adam':
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate) 
    elif optimizer == "rmsprop":
        opt = tf.keras.optimizers.RMSprop(learning_rate=learning_rate) 
    else:
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate,momentum=0.9)
    
    model.compile(optimizer=opt, loss='binary_crossentropy',metrics=['accuracy']) 
    
    return model

In [ ]:
tuner = kt.BayesianOptimization(hypermodel = build_model, 
                                objective='val_accuracy', 
                                max_trials = 20, 
                                num_initial_points=5, 
                                #random 
                                seed = 10, 
                                directory = 'keras_tuner_logs', 
                                project_name = "misson_ai_sekho" 
                               )


In [ ]:
tuner.search_space_summary()

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5)

In [ ]:
tuner.search(X_train_scaled, y_train, 
             epochs=50, 
             batch_size=32, 
             validation_split = 0.20, 
             callbacks=[stop_early] 
            )



In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
best_hps

In [ ]:

for i in range(best_hps.get('n_layers')): 
    print(f" Layer {i+1} units : {best_hps.get(f'units_{i}')}") 
    print(f" Layer {i+1} activation : {best_hps.get(f'activation_{i}')}") 
    print(f" Layer {i+1} dropout : {best_hps.get(f'dropout_{i}')}") 
    print(f" Layer {i+1} L2 : {best_hps.get(f'lr_{i}')}") 
    print(f" Optimizer : {best_hps.get(f'optimizer')}") 
    print(f" Learning Rate : {best_hps.get(f'learning_rate')}")



In [ ]:
best_model = tuner.hypermodel.build(best_hps)

history = best_model.fit( X_train_scaled, y_train, epochs=200, batch_size=32, validation_split = 0.20, 
                         callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,
                                                                     restore_best_weights=True)], )

loss, accuracy = best_model.evaluate(X_test_scaled, y_test) 
print(f"loss : {loss}") 
print(f"accuracy : {accuracy}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4)) 
plt.plot(history.history['loss'], label='Training Loss') 
plt.plot(history.history['val_loss'], label='Validation Loss') 
plt.grid(alpha=0.4) 
plt.legend() 
plt.xlabel("Epoch") 
plt.ylabel("Loss (bayesian)")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4)) 
plt.plot(history.history['accuracy'], label='Training accuracy') 
plt.plot(history.history['val_accuracy'], label='Validation accuracy') 
plt.grid(alpha=0.4) 
plt.legend() 
plt.xlabel("Epoch") 
plt.ylabel("Loss (bayesian)")



In [ ]:
best_model.save("best_churn_model.keras")

In [ ]:
with open('selected_feature.pkl','wb') as f: 
    pickle.dump(selected_features,f)

In [ ]:
#Prediction
def load_model_comp(): 
    model = tf.keras.models.load_model('/kaggle/working/best_churn_model.keras')

    #load scaler 
    with open('scaler.pkl','rb') as f: 
        scaler = pickle.load(f)

    with open('selected_feature.pkl','rb') as f: 
        selected_features = pickle.load(f)

    return model, scaler, selected_features

In [ ]:
X[selected_features].head(3)

In [ ]:
def pred(): 
    model,scaler, selected_feature = load_model_comp()

    sample_customer = pd.DataFrame([[34, 1, 0, 56.78, 2300]],columns=selected_features) 
    sample_scaled = scaler.transform(sample_customer) 
    prediction = model.predict(sample_scaled)[0][0] 
    print(f"Churn prob:{prediction}") 
    print(f"Will churn :{'yes' if prediction> 0.5 else 'no'}")

pred()